# 🚀 Модуль 5. Урок 6. Статистические ловушки в бизнесе: парадокс Симпсона, регрессия к среднему и подглядывание

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_05_statistics/lesson_06_statistical_traps_in_business.ipynb)

## 📌 Бизнес-контекст и методология
Самые дорогие ошибки бизнеса совершаются из-за неверной трактовки статистики:
1. **Парадокс Симпсона**: тренд, наблюдаемый в отдельных группах, обращается вспять при объединении данных.
2. **Peeking Problem**: ежедневная проверка A/B теста до набора выборки увеличивает ложную значимость в 3-5 раз.
3. **Регрессия к среднему**: после аномально успешного месяца следующий почти наверняка будет ближе к норме.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Демонстрация проблемы подглядывания (Peeking Problem)
import numpy as np
from scipy import stats

np.random.seed(42)
n_simulations = 500
false_positives = 0

# Проверяем две группы БЕЗ реального эффекта (A/A тест)
for _ in range(n_simulations):
    a = np.random.normal(0, 1, 1000)
    b = np.random.normal(0, 1, 1000)
    
    # Подглядываем на шагах 100, 200, 500, 1000
    for step in [100, 200, 500, 1000]:
        t_stat, p_val = stats.ttest_ind(a[:step], b[:step])
        if p_val < 0.05:
            false_positives += 1
            break # Остановили тест раньше времени!

observed_alpha = false_positives / n_simulations
print(f"Номинальный уровень ошибки alpha:  5.0%")
print(f"Реальный уровень ложных тревог:    {observed_alpha:.1%}")


Номинальный уровень ошибки alpha:  5.0%
Реальный уровень ложных тревог:    14.0%


### 🎯 Практическое задание 1: Оценка завышения ошибки I рода
Во сколько раз подглядывание завысило вероятность ложной победы?

In [3]:
### ВАШ КОД ЗДЕСЬ
inflation = observed_alpha / 0.05
print(f"Риск ложного вывода вырос в {inflation:.1f} раз!")

assert observed_alpha > 0.05, "Подглядывание всегда завышает alpha"
print("✅ Тест успешно пройден: опасность Peeking Problem доказана эмпирически!")


Риск ложного вывода вырос в 2.8 раз!
✅ Тест успешно пройден: опасность Peeking Problem доказана эмпирически!


### 💡 Как защититься от подглядывания в A/B тестах?
1. Заранее фиксировать размер выборки через калькулятор мощности (Sample Size).
2. Применять последовательный анализ (Sequential Testing / Always Valid p-values).